In [ ]:
%pip install -qqq pandas numpy scikit-learn sentence-transformers lightgbm matplotlib seaborn Janome 
# tqdm

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

from janome.tokenizer import Tokenizer
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import HashingVectorizer, TfidfVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
# from tqdm import tqdm

In [ ]:
seed = 42
y_col_name = "isfake"

## 1. データ読み込み

In [ ]:
url = "https://github.com/tanreinama/Japanese-Fakenews-Dataset/raw/refs/heads/master/fakenews.csv"

df_raw = pd.read_csv(url)

In [ ]:
df_raw.head(5)

In [ ]:
len(df_raw["id"])

## 2. 前処理（わかち書き）の定義

In [ ]:
t = Tokenizer()

In [ ]:
def ja_tokenizer(text):
    # エラー防止：NaNや数値が混ざっていたら文字列に変換
    text = str(text) 
    
    return [token.surface for token in t.tokenize(text)]

## 3. モデル学習と評価

In [ ]:
# 動作確認用
df_source = df_raw[:100].copy()

# 本番用
# df_source = df_raw.copy()

In [ ]:
# データリーク防止のために、先にデータセットを分ける
# 残差分析の為に、df_source["context"]だけ取り出して分割ではなく、全体を分割、

X_train, X_test, y_train, y_test= train_test_split(
    df_source.drop(y_col_name, axis=1), 
    # df_source["context"], 
    df_source[y_col_name], 
    test_size=0.2, 
    stratify=df_source[y_col_name],
    random_state=seed
)

# 残差分析の為にX_testをcopyしておく
df_residual = X_test.copy()
df_residual["y_true"] = y_test

### 3.1 TfidfVectorizer

In [ ]:
# CPUの枯渇対策
# 辞書を作らないハッシュベクトライザーを定義

hash_vectorizer = HashingVectorizer(
    tokenizer=ja_tokenizer,
    n_features=2**18,
    alternate_sign=False
)

tfidf_transformer = TfidfTransformer()

X_train_hashed = hash_vectorizer.transform(X_train["context"])
X_train_vector = tfidf_transformer.fit_transform(X_train_hashed)

X_test_hashed = hash_vectorizer.transform(X_test["context"])
X_test_vector = tfidf_transformer.transform(X_test_hashed)

In [ ]:
# contextのベクトル化
# TfidfVectorizerの場合

# vectorizer = TfidfVectorizer(analyzer=ja_tokenizer)

# X_train_vector = np.array(vectorizer.fit_transform(X_train["context"]).toarray().tolist())

# X_test_vector = vectorizer.transform(X_test["context"]) #trainデータでfit_transfromした物をそのまま使う

In [ ]:
model_lr = LogisticRegression(
    # multi_class="multinomial", 
    solver="lbfgs",      # multinomialに対応した最適化アルゴリズム
    random_state=seed
)

In [ ]:
model_lr.fit(X_train_vector, y_train)

In [ ]:
y_pred_lr = model_lr.predict(X_test_vector)
# y_pred_lr = model_lr.predict_proba(X_test_vector)　#確率求める時はこっち

In [ ]:
y_pred_lr

In [ ]:
# 残差分析用に予測結果を保存
df_residual["y_pred_lr"] = y_pred_lr

In [ ]:
df_residual.head(3)

In [ ]:
# 評価指標の計算
acc_lr = accuracy_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr, average="weighted")

In [ ]:
print(f"tfid + lr:\nacc:{acc_lr}\nf1:{f1_lr}")

In [ ]:
# 混合行列を求める
classes = sorted(df_source["isfake"].unique())

cm_lr = confusion_matrix(y_test, y_pred_lr, labels=classes)
df_cm_lr = pd.DataFrame(cm_lr, index=classes, columns=classes)


In [ ]:
# 混合行列を描画

plt.figure(figsize=(6, 5))
sns.heatmap(df_cm_lr, annot=True, fmt="d", cmap="Blues", cbar=True)

plt.title("Confusion Matrix")
plt.xlabel("pred")
plt.ylabel("true")

plt.tight_layout()
plt.show()

In [ ]:
dict_report_tfid_lr = classification_report(y_test, y_pred_lr, output_dict=True)

In [ ]:
# accuracy行や全体の集計を扱いやすくするため調整
df_report_lr = pd.DataFrame(dict_report_tfid_lr).iloc[:-1, :].T 

plt.figure(figsize=(8, 5))
sns.heatmap(
    df_report_lr,
    annot=True,  # 数値をマスの中に表示
    cmap="Blues",  # 青系のカラーマップ
    fmt=".2f",  # 小数点以下2桁まで表示
    cbar=True,  # カラーバーを表示
)

plt.title("lr: Classification Report")
plt.show()

## 3.2 sentencetransformer + lightgbm

In [ ]:
# embedding用のモデルを設定

# 動作確認
model_st = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# 本番用
# model_st = SentenceTransformer(
#     "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
#     device="cuda"
# )

In [ ]:
X_train_encoded = model_st.encode(
    X_train["context"].tolist(),
    batch_size=32,            # 一度に処理する件数を32件に制限（メモリ対策）
    show_progress_bar=True, #進捗barの表示
)


X_test_encoded = model_st.encode(
    X_test["context"].tolist(),
    batch_size=32,            # 一度に処理する件数を32件に制限（メモリ対策）
    show_progress_bar=True, #進捗barの表示
)

# X_train_encoded = model.encode(X_train.tolist())
# X_test_encoded = model.encode(X_test.tolist())

In [ ]:
model_lgb = lgb.LGBMClassifier(
    boosting_type="gbdt",
    n_estimators=100,
    learning_rate=0.1,
    # is_unbalance=True, 
    importance_type="gain",
    verbose = -1,
    random_state=seed
)

In [ ]:
model_lgb.fit(X_train_encoded, y_train)

In [ ]:
y_pred_lgb = model_lgb.predict(X_test_encoded)
# y_pred_lgb = model_lgb.predict_proba(X_test_encoded) #確率でやる時はこっち

In [ ]:
# 残差分析用に予測結果を保存
df_residual["y_pred_lgb"] = y_pred_lgb

In [ ]:
df_residual.head(5)

In [ ]:
# 評価指標の計算
acc_lgb = accuracy_score(y_test, y_pred_lgb)
f1_lgb = f1_score(y_test, y_pred_lgb, average="weighted")
print(f"st + lgb:\nacc:{acc_lgb}\nf1:{f1_lgb}")

In [ ]:
cm_lgb = confusion_matrix(y_test, y_pred_lgb, labels=classes)
df_cm_lgb = pd.DataFrame(cm_lgb, index=classes, columns=classes)

# 描画設定
plt.figure(figsize=(6, 5))
sns.heatmap(df_cm_lgb, annot=True, fmt="d", cmap="Blues", cbar=True)

# ラベルとタイトル
plt.title("Confusion Matrix")
plt.xlabel("pred")
plt.ylabel("true")

# 表示
plt.tight_layout()
plt.show()

In [ ]:
dict_report_st_lgb = classification_report(y_test, y_pred_lgb, output_dict=True)


In [ ]:
# accuracy行や全体の集計を扱いやすくするため調整
df_report_lgb = pd.DataFrame(dict_report_st_lgb).iloc[:-1, :].T 

plt.figure(figsize=(8, 5))
sns.heatmap(
    df_report_lgb,
    annot=True,  # 数値をマスの中に表示
    cmap="Blues",  # 青系のカラーマップ
    fmt=".2f",  # 小数点以下2桁まで表示
    cbar=True,  # カラーバーを表示
)

plt.title("lgb: Classification Report")
plt.show()

## 4. 精度の分析

## 4.1 データ準備

In [ ]:
# 本番環境での計算結果は保存する

# output_path = "result.csv"
# df_residual.to_csv(output_path, index= False, encoding="utf-8")


In [ ]:
df_residual.head(2)

In [ ]:
# contextの文字数を求める
df_residual["context_len"] = df_residual["context"].str.len()

# contextの何割がfakeかを計算する
df_residual["fake_ratio"] = df_residual["nchar_fake"]/df_residual["context_len"]

In [ ]:
df_residual.head(2)

In [ ]:
# 予測したクラスがあっているかどうかの確認フラグの作成
df_residual["tfid_lr_pred_correct_flag"] = df_residual["y_pred_lr"] == df_residual["y_true"]
df_residual["st_lgb_pred_correct_flag"] = df_residual["y_pred_lgb"] == df_residual["y_true"]

In [ ]:
df_residual.head(2)

In [ ]:
# context_lenの文字数を離散化する
df_residual["context_len_range"] = (df_residual["context_len"]//100)*100

# fake_ratioを離散化する

df_residual["fake_ratio_range"] = df_residual["fake_ratio"].round(1)

In [ ]:
df_residual.head(10)

## 4.2 データの分布の確認

## 4.2.1 tdif+lr

In [ ]:
# コンテキストの文字数と、正解不正解の関係
cross_table_context_len_correctratio_lr = pd.crosstab(
                                                df_residual["context_len_range"],
                                                df_residual["tfid_lr_pred_correct_flag"],
                                                normalize="index"
                                              )

In [ ]:
cross_table_context_len_correctratio_lr.plot(
                                            kind="bar", 
                                            stacked=True, 
                                            cmap="crest", 
                                            edgecolor="black", 
                                            width=0.6
                                        )

plt.title("TF-IDF + Logistic Regression:\
            Prediction Correctness by Text Length")
plt.xlabel("context_len")
plt.ylabel("Proportion")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper center")
plt.show()

In [ ]:
# コンテキストのフェイクニュースの割合と、正解不正解の関係
cross_table_fakeratio_correctratio_lr = pd.crosstab(
                                                    df_residual["fake_ratio_range"],
                                                    df_residual["tfid_lr_pred_correct_flag"],
                                                    normalize="index"
                                                  )

In [ ]:
cross_table_fakeratio_correctratio_lr.plot(
                                            kind="bar", 
                                            stacked=True, 
                                            cmap="crest", 
                                            edgecolor="black", 
                                            width=0.6
                                        )

plt.title("TF-IDF + Logistic Regression:\
Prediction Correctness by Fake Content Ratio")
plt.xlabel("fake_ratio")
plt.ylabel("Proportion")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper center")
plt.show()

## 4.2.2 sentencetranceformer + lgb

In [ ]:
# コンテキストの文字数と、正解不正解の関係
cross_table_context_len_correctratio_lgb = pd.crosstab(
                                                    df_residual["context_len_range"],
                                                    df_residual["st_lgb_pred_correct_flag"],
                                                    normalize="index"
                                                  )

In [ ]:
cross_table_context_len_correctratio_lgb.plot(
                                            kind="bar", 
                                            stacked=True, 
                                            cmap="crest", 
                                            edgecolor="black", 
                                            width=0.6
                                        )

plt.title("SentenceTransformer + LightGBM:\
Prediction Correctness by Text Length")
plt.xlabel("context_len")
plt.ylabel("Proportion")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper center")
plt.show()

In [ ]:
# コンテキストのフェイクニュースの割合と、正解不正解の関係
cross_table_fakeratio_correctratio_lgb = pd.crosstab(
                                                    df_residual["fake_ratio_range"],
                                                    df_residual["st_lgb_pred_correct_flag"],
                                                    normalize="index"
                                                  )

In [ ]:
cross_table_fakeratio_correctratio_lgb.plot(
                                            kind="bar", 
                                            stacked=True, 
                                            cmap="crest", 
                                            edgecolor="black", 
                                            width=0.6
                                        )

plt.title("SentenceTransformer + LightGBM:\
Prediction Correctness by Fake Content Ratio")
plt.xlabel("fake_ratio")
plt.ylabel("Proportion")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper center")
plt.show()

## 5. 追加実験

tfid+logisticregression方が精度が良かったので、下記の組み合わせでも試してみる。

・sentencetransformer + logisticregression  
・tfid+lightgbm  
の精度も確認する。

## 5.1 sentence transformer + logisticregression 

In [ ]:
model_lr_st = LogisticRegression(
    solver="lbfgs",
    random_state=seed
)

In [ ]:
model_lr_st.fit(X_train_encoded, y_train)

In [ ]:
y_pred_lr_st = model_lr_st.predict(X_test_encoded)

# 評価指標の計算
acc_lr_st = accuracy_score(y_test, y_pred_lr_st)
f1_lr_st = f1_score(y_test, y_pred_lr_st, average="weighted")

print(f"st + lr:\nacc:{acc_lr_st}\nf1:{f1_lr_st}")

## 5.2 tfid + lightgbm

In [ ]:
model_lgb_tfid = lgb.LGBMClassifier(
    boosting_type="gbdt",
    n_estimators=100,
    learning_rate=0.1,
    # is_unbalance=True, 
    # importance_type="gain",
    verbose = -1,
    random_state=seed
)

In [ ]:
model_lgb_tfid.fit(X_train_vector, y_train)

In [ ]:
y_pred_lgb_tfid = model_lgb_tfid.predict(X_test_vector)

acc_lgb_tfid = accuracy_score(y_test, y_pred_lgb_tfid)
f1_lgb_tfid = f1_score(y_test, y_pred_lgb_tfid, average="weighted")
print(f"tfid+lgb:\nacc:{acc_lgb_tfid}\nf1:{f1_lgb_tfid}")